## IMPORT

In [1]:
from selenium import webdriver
import pandas as pd
import time
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, StaleElementReferenceException, NoSuchElementException, WebDriverException

## GET ALL RESTAURANT LINKS

In [ ]:
driver = webdriver.Chrome()
driver.get('https://pergikuliner.com/restaurants?utf8=%E2%9C%93&search_place=&default_search=Jakarta&search_name_cuisine=&commit=')

In [ ]:
while True:
    try:
        # klik button 'Hasil Pencarian Berikutnya..'
        button = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.ID, 'next')))
        button.click()
        print("Klik")

        # WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, 'item-name')))
        
    except StaleElementReferenceException:
        print("Stale")
        continue

    except TimeoutException:
        print("Finish")
        break

restaurant_list = []
for restaurant in driver.find_elements('xpath', '//h3[@class="item-name"]'):
    restaurant_clean = restaurant.text.replace(' ', '-').replace("'",'-').replace("%",'').replace("!",'').replace(".",'-').replace('&','dan').replace('---','-')
    restaurant_list.append(restaurant_clean)

# 's jadi -s
# %Arabica jadi Arabica
# holycow! jadi holycow
# n.o.b jadi n-o-b
# pancake co. by dore jadi pancake-co-by

# manual check
# ' jadi -
# -- jadi -
# (PHD) jadi PHD, semua () hilangin
# mama(m) jadi mama-m
# aged + butchered jadi aged-butcered
# strada+ coffee jdi strada-coffee
# 774,https://pergikuliner.com/restaurants/jakarta/Kopi@Kopi-Ground-Zero-Kelapa-Gading


location_list = []
for i in range(1, 1500):  # assume 1500 restaurant
    try:
        xpath = f'//*[@id="restaurant_contents"]/div[{i}]/div[1]/div/div'
        location = driver.find_element(By.XPATH, xpath)
        location_list.append(location.text.split('|')[0].strip().replace(' ','-'))
    except Exception as e:
        break

data_tuples = list(zip(restaurant_list[:],location_list[:]))
temp_df = pd.DataFrame(data_tuples, columns=['Restaurant', 'Location'])
temp_df
driver.close()

In [ ]:
# a list containing all restaurants url
link = 'https://pergikuliner.com/restaurants/jakarta/'
link_list = []
for (a,b) in zip(restaurant_list, location_list):
    complete_link = link + a + '-'+ b
    link_list.append(complete_link)

link_list

In [ ]:
# save data : restaurant name and location
temp_df.to_csv('data.csv')

In [ ]:
# save restaurant link
df = pd.DataFrame(link_list)
df.to_csv('data_link.csv')

## ITERATE ALL RESTAURANT LINKS

In [2]:
data = pd.read_csv('data_link.csv')
data = data['0']
data = data.values.tolist()

In [3]:
driver = webdriver.Chrome()

restaurant_list = []
cuisine_list = []
rating_list = []
location_list = []
price_list = []
username_list = []
user_rating_list = []
review_list = []

for i in data[1400:1498]:
    driver.get(i)

    # Restaurant name
    for element in driver.find_elements('xpath', '//div[@class="heading"]'):
        restaurant_clean = element.text.split("[")[0].strip()
        if restaurant_clean: # krn output pertama bakal empty line, jd ksh ini
            restaurant_name = restaurant_clean

    # Cuisine type
    for element in driver.find_elements('xpath', '//span[@class="cuisine-type"]'):
        cuisine = element.text.strip()
        if cuisine:
            cuisine = cuisine.replace('[',"").replace(']',"").strip()
            cuisine_clean = cuisine

    # Rating overall
    # rating = driver.find_elements('xpath', '//div[@class="item-rating best-rating"]')[1].text

    # Rating overall
    try:
        rating_elements = driver.find_elements(By.XPATH, '//div[@class="item-rating best-rating"]')
        if len(rating_elements) > 1:
            rating = rating_elements[1].text
        else:
            rating_elements = driver.find_elements(By.XPATH, '//div[@class="item-rating good-rating"]')
            if len(rating_elements) > 1:
                rating = rating_elements[1].text
            else:
                rating = "N/A"  # Default if no rating is available
    except Exception as e:
        rating = "N/A"

    
    # Location
    location = driver.find_element('xpath', '/html/body/div[1]/main/div[2]/div/div/div[1]/span[4]/a/span').text

    # Price
    # price = driver.find_element('xpath', '/html/body/div[1]/main/div[2]/div/div/div[2]/div[3]/div[3]/article/p[5]/span[1]').text

    # Price
    price = "N/A"
    try:
        price_element = driver.find_element(By.XPATH, '/html/body/div[1]/main/div[2]/div/div/div[2]/div[3]/div[3]/article/p[5]/span[1]')
        price = price_element.text
    except NoSuchElementException:
        try:
            price_element = driver.find_element(By.XPATH, '/html/body/div[1]/main/div[2]/div/div/div[2]/div[3]/div[3]/article/p[4]/span[1]')
            price = price_element.text
        except NoSuchElementException:
            print("Price not found for this page.")


    # load button 
    while True:
        try:
            button = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.ID, 'next_review')))
            button.click()
        except StaleElementReferenceException:
            continue
        except TimeoutException:
            break

    # Username, user rating, and review (200 max reviews)
    for j in range(1, 200):
        try:
            # Username
            username_xpath = f"/html/body/div[1]/main/div[2]/div/div/div[5]/div[4]/div[{j}]/div[1]/figure/a"
            username = driver.find_element(By.XPATH, username_xpath).get_attribute('href').split('.com/')[-1]

            # User rating
            user_rating_xpath = f"/html/body/div[1]/main/div[2]/div/div/div[5]/div[4]/div[{j}]/div[2]/span/span/span"
            user_rating = driver.find_element(By.XPATH, user_rating_xpath).text.strip()

            # Review
            review_title_xpath = f"/html/body/div[1]/main/div[2]/div/div/div[5]/div[4]/div[{j}]/div[2]/h3/a"
            review_content_xpath = f"/html/body/div[1]/main/div[2]/div/div/div[5]/div[4]/div[{j}]/div[2]/div[1]/div/p"

            review_title = driver.find_element(By.XPATH, review_title_xpath).text.strip()
            review_content = driver.find_element(By.XPATH, review_content_xpath).text.strip()
            review = f"{review_title}\n{review_content}"

            # Append data
            restaurant_list.append(restaurant_name)
            cuisine_list.append(cuisine_clean)
            rating_list.append(rating)
            location_list.append(location)
            price_list.append(price)
            username_list.append(username)
            user_rating_list.append(user_rating)
            review_list.append(review)
        except Exception as e:
            break

# gabung jadi data frame
data_tuples = list(zip(restaurant_list, cuisine_list, rating_list, location_list, price_list, username_list, user_rating_list, review_list))
temp_df = pd.DataFrame(data_tuples, columns=['Restaurant Name', 'Cuisine Type', 'Restaurant Rating', 'Location', 'Price', 'Username', 'User Rating', 'User Review'])
driver.close()

In [4]:
# 1400-1497
temp_df.to_csv('master_data46.csv')

combine all to final_data.csv

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
files = ['master_data.csv', 'master_data1.csv', 'master_data2.csv',
         'master_data3.csv', 'master_data4.csv', 'master_data5.csv',
         'master_data6.csv', 'master_data7.csv', 'master_data8.csv',
         'master_data9.csv', 'master_data10.csv', 'master_data11.csv',
         'master_data12.csv', 'master_data13.csv', 'master_data14.csv',
         'master_data15.csv', 'master_data16.csv', 'master_data17.csv',
         'master_data18.csv', 'master_data19.csv', 'master_data20.csv',
         'master_data21.csv', 'master_data22.csv', 'master_data23.csv',
         'master_data24.csv', 'master_data25.csv', 'master_data26.csv',
         'master_data27.csv', 'master_data28.csv', 'master_data29.csv',
         'master_data30.csv', 'master_data31.csv', 'master_data32.csv',
         'master_data33.csv', 'master_data34.csv', 'master_data35.csv',
         'master_data36.csv', 'master_data37.csv', 'master_data38.csv',
         'master_data39.csv', 'master_data40.csv', 'master_data41.csv',
         'master_data42.csv', 'master_data43.csv', 'master_data44.csv',
         'master_data45.csv', 'master_data46.csv'
        ]

In [ ]:
df = pd.concat([pd.read_csv(file) for file in files], ignore_index=True)

In [ ]:
df.drop(columns='Unnamed: 0', inplace=True)

In [ ]:
df.to_csv('final_data.csv')

### LIST DATA DONE

In [22]:
# 0-9
# temp_df.to_csv('master_data.csv')
# 10-19
# temp_df.to_csv('master_data1.csv')
# 20-49
# temp_df.to_csv('master_data2.csv')
# 50-99
# temp_df.to_csv('master_data3.csv')
# 100-130
# temp_df.to_csv('master_data4.csv')
# 131 skip no review
# 132-149
# temp_df.to_csv('master_data5.csv')
# 150-165
# temp_df.to_csv('master_data6.csv')
# 166 FIXED
# 167-199
# temp_df.to_csv('master_data7.csv')
# 200-220
# temp_df.to_csv('master_data8.csv')
# 221-222
# temp_df.to_csv('master_data9.csv')
# 223 FIXED
# 224-236
# temp_df.to_csv('master_data10.csv')
# 237 error sdh tutup
# 238-249
# temp_df.to_csv('master_data11.csv')
# 250-255
# temp_df.to_csv('master_data12.csv')
# 256 ramen ya tutup
# 257-261
# temp_df.to_csv('master_data13.csv')
# 262 FIXED
# 263-299
# temp_df.to_csv('master_data14.csv')
# 298-314
# temp_df.to_csv('master_data15.csv')
# 315 FIXED
# 316-329
# temp_df.to_csv('master_data16.csv')
# 330-349
# temp_df.to_csv('master_data17.csv')
# 350-399
# temp_df.to_csv('master_data18.csv')
# 400-453
# temp_df.to_csv('master_data19.csv')
# 454-466
# temp_df.to_csv('master_data20.csv')
# 467 GADA REVIEW
# 468-514
# temp_df.to_csv('master_data21.csv')
# 515 TUTUP GADA REVIEW
# 516-599
# temp_df.to_csv('master_data22.csv')
# 600-653
# temp_df.to_csv('master_data23.csv')
# 654 GADA REVIEW
# 655-699
# temp_df.to_csv('master_data24.csv')
# 700
# temp_df.to_csv('master_data25.csv')
# 701 GADA REVIEW
# 702-729
# temp_df.to_csv('master_data26.csv')
# 730 GADA REVIEW
# 731-738
# temp_df.to_csv('master_data27.csv')
# 739 GADA REVIEW
# 740-778
# temp_df.to_csv('master_data28.csv')
# 779 GADA REVIEW
# 780-794
# temp_df.to_csv('master_data29.csv')
# 795 GADA REVIEW
# 796-799
# temp_df.to_csv('master_data30.csv')
# 800-802
# temp_df.to_csv('master_data31.csv')
# 804-870
# temp_df.to_csv('master_data32.csv')
# 872-899
# temp_df.to_csv('master_data33.csv')
# 900-999
# temp_df.to_csv('master_data34.csv')
# 1000 GADA REVIEW
# 1001-1060
# temp_df.to_csv('master_data35.csv')
# 1061 SAMA KAYAK SBLM
# 1062-1085
# temp_df.to_csv('master_data36.csv')
# 1086 GADA
# 1087-1095
# temp_df.to_csv('master_data37.csv')
# 1096 GADA
# 1097-1174
# temp_df.to_csv('master_data38.csv')
# 1175 GADA
# 1176-1210
# temp_df.to_csv('master_data39.csv')
# 1211 GADA
# 1212-1256
# temp_df.to_csv('master_data40.csv')
# 1257 GADA
# 1258-1298
# temp_df.to_csv('master_data41.csv')
# 1299 GADA
# 1300-1315
# temp_df.to_csv('master_data42.csv')
# 1316 GADAA
# 1317-1346
# temp_df.to_csv('master_data43.csv')
# 1347 GADA
# 1348-1368
# temp_df.to_csv('master_data44.csv')
# 1369 GADA
# 1370-1399
# temp_df.to_csv('master_data45.csv')